<a href="https://colab.research.google.com/github/Yoram-Jacobs/DressAppV1/blob/main/notebooks/dressapp_eyes_lora_v5_cultural_attire.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# 👘 DressApp Eyes LoRA v5: Cultural, Traditional & Religious Garments & Headwear
### Multilingual Fine-Tuning (Qwen2.5-VL-3B-Instruct) + QLoRA + llama.cpp GGUF Quantization (`Q4_K_M`)

**DressApp Eyes** (`Qwen2.5-VL-3B-Instruct`) powers the on-premise vision inference server (`dressapp-eyes` container) on the production Hetzner VPS (`178.105.144.142`). While standard fashion datasets focus primarily on western basics (T-shirts, jeans, sneakers, suits), global users wear cultural, religious, and traditional attire and headwear.

---

### 🎯 LoRA v5 Objectives:
1. **Cultural & Religious Garments:**
   - **Galabiya** (جلابية / ג'לביה): Middle Eastern/North African full-length tunic/robe, collarless/slit neck, long sleeves (`Full Body`).
   - **Kaftan** (قفطان / קפטן): Flowing ornate robe/tunic, zari/metallic embroidery, wide/bell sleeves (`Full Body`).
   - **Thobe / Dishdasha** (ثوب / دشداشة): Ankle-length tailored tunic with mandarin collar and cuffed sleeves (`Full Body`).
   - **Abaya** (عباية / עבאיה): Floor-length loose modest cloak, front-open or pullover (`Full Body` / `Outerwear`).
   - **Kurta / Kurti** (कुर्ता / קורטה): Straight collarless/mandarin tunic worn over churidar or pyjama (`Top`).
   - **Sherwani** (शेरवानी): Structured formal jacket/coat with stand collar, brocade/jacquard (`Outerwear`).
   - **Sari** (साड़ी / סארי): Draped unstitched fabric with decorated pallu and matching choli blouse (`Full Body`).
   - **Hanbok** (한복): Traditional Korean jeogori jacket with goreum ribbons and billowing chima skirt (`Full Body`).
   - **Kimono / Yukata** (着物 / 浴衣): Japanese wrapped-front T-shaped robe with wide sleeves and obi sash (`Full Body`).
   - **Dirndl** (דירנדל): Alpine folk dress with fitted bodice, blouse, full skirt, and apron (`Full Body`).
   - **Guayabera** (גואיאברה): Caribbean/Latin American four-pocket pleated dress shirt with alforzas (`Top`).
2. **Traditional Headwear & Religious Accessories (`category: "Accessories"`):**
   - **Kippah / Yarmulke** (כיפה / יארמולקה): Jewish traditional skullcap in velvet, satin, or knitted/crochet (`sub_category: "Headwear"`).
   - **Keffiyeh / Shemagh** (كوفية / شماغ / כאפייה): Middle Eastern checkered woven headscarf or shoulder drape with tassels (`sub_category: "Headwear"` or `"Scarves & Wraps"`).
   - **Turban / Dastar / Pagri** (عمامة / पगड़ी / טורבן): Draped pleated fabric head covering across Sikh, Middle Eastern, and South Asian traditions (`sub_category: "Headwear"`).
   - **Hijab / Shayla** (حجاب / שיילה): Modest lightweight hair and neck wrap in chiffon or jersey (`sub_category: "Headwear"` / `"Scarves & Wraps"`).
   - **Shtreimel** (שטריימל): Traditional Hasidic velvet-crown circular fur hat worn on Shabbat and festivals (`sub_category: "Headwear"`).
   - **Fez / Tarboosh** (طربوش / תרבוש): Cylindrical red felt hat with black tassel (`sub_category: "Headwear"`).
3. **New Pattern Recognition:** Full visual grounding for `pattern: "embroidered"` (intricate needlework, thread work, metallic zari).
4. **Multilingual Invariance across all 13 Languages:** English (`en`), Hebrew (`he`), Arabic (`ar`), Spanish (`es`), French (`fr`), German (`de`), Italian (`it`), Portuguese (`pt`), Dutch (`nl`), Russian (`ru`), Chinese (`zh`), Japanese (`ja`), Hindi (`hi`).
5. **Zero-Regression Western Basics:** Blended with the baseline 5,000-item everyday wardrobe dataset to retain high accuracy on standard wardrobe items.
6. **GGUF Export & 1-Click Deployment:** Merge LoRA into base FP16, quantize with `llama.cpp` to `Q4_K_M`, test against cultural samples, and push directly to VPS via SCP.


## 🛠️ Step 1: Environment Setup & Tooling
Install deep learning dependencies, Qwen VL utilities, and compile `llama.cpp` conversion and quantization tools.


In [ ]:
# Check GPU availability (Run on T4, L4, or A100 GPU)
!nvidia-smi

import os
os.environ['USE_TF'] = '0'
os.environ['USE_TORCH'] = '1'

# Install Transformers, PEFT, BitsAndBytes, and Qwen VL Utilities
!pip install -q -U 'transformers>=4.49.0' accelerate peft 'bitsandbytes>=0.46.1' qwen-vl-utils gguf sentencepiece protobuf paramiko scp datasets

# Clone and build llama.cpp for GGUF conversion & quantization utilities
if not os.path.exists('llama.cpp'):
    os.system('git clone https://github.com/ggerganov/llama.cpp.git')
    os.system('cmake -B llama.cpp/build -S llama.cpp -DGGML_CUDA=OFF')
    os.system('cmake --build llama.cpp/build --config Release -j$(nproc) --target llama-quantize')

print('✅ Environment ready and llama.cpp tools compiled!')


## 📥 Step 2: (Optional) Verify Connection to Hetzner VPS Server
Tests SSH connectivity to the production VPS (`178.105.144.142`). If not yet authorized, your public key will be displayed so you can paste it into the chat.


In [ ]:
import os, subprocess
from pathlib import Path

# 1. Ensure SSH key exists
Path('/root/.ssh').mkdir(parents=True, exist_ok=True)
if not os.path.exists('/root/.ssh/id_ed25519'):
    os.system("ssh-keygen -t ed25519 -N '' -f /root/.ssh/id_ed25519")

# 2. Test SSH connection
print('Testing SSH connection to Hetzner VPS (178.105.144.142)...')
try:
    proc = subprocess.run(
        ['ssh', '-o', 'BatchMode=yes', '-o', 'ConnectTimeout=5', '-o', 'StrictHostKeyChecking=no', 'root@178.105.144.142', 'echo connected'],
        capture_output=True, text=True, timeout=8
    )
    is_connected = ('connected' in proc.stdout)
except Exception:
    is_connected = False

if is_connected:
    print('✅ Hetzner VPS SSH connection verified!')
else:
    print('ℹ️ Colab SSH Key is not yet authorized on the VPS.')
    print('To authorize 1-click datacenter download/upload, copy and paste this key into your chat:')
    pub_key_path = Path('/root/.ssh/id_ed25519.pub')
    if pub_key_path.exists():
        print(pub_key_path.read_text().strip())


## 📦 Step 3: Hybrid Ingestion: Real Hugging Face Items + Cultural Attire & Headwear Distillation
This step builds the comprehensive **LoRA v5 Multilingual Training Dataset** by combining:
1. **Baseline Everyday Wardrobe:** 5,000 items from `dataset_distilled.tar.gz` (T-shirts, jeans, sneakers, coats, bags).
2. **Real Photographic Cultural Garments from Hugging Face:**
   - **`ashraq/fashion-product-images-small`**: Real studio cutout e-commerce photos on pure white backgrounds (~2,780 real products: Kurtas, Sarees, Kurtis, Dupattas, Kurta Sets, Salwars, Churidars).
   - **`EDDOUM/moroccan_caftan`**: Real high-resolution photographic Moroccan Kaftans with silk, brocade, crepe, and gold *skalli* embroidery.
3. **Targeted Distillation for Traditional Attire & Headwear:**
   - **Cultural Robes & Apparel:** Galabiya, Thobe, Abaya, Dirndl, Guayabera, Hanbok, Kimono.
   - **Traditional & Religious Headwear (`Accessories` -> `Headwear`):** Kippah/Yarmulke (כיפה), Keffiyeh/Shemagh (كوفية/שמאג), Turban/Dastar/Pagri (عمامة/पगड़ी/טורבן), Hijab/Shayla (حجاب), Shtreimel (שטריימל), Fez/Tarboosh (طربوش).
   - Grounded across all **13 DressApp languages** (en, he, ar, es, fr, de, it, pt, nl, ru, zh, ja, hi).
4. **Combined Output:** Exports `curated_dataset_v5.jsonl` with balanced representation across apparel, accessories, and headwear.


In [ ]:
import os, sys, json, tarfile, random
from pathlib import Path

# Resilient Pillow loader to handle Colab kernel module cache collisions
try:
    from PIL import Image, ImageDraw
except (ImportError, AttributeError):
    import subprocess
    print('🔄 Resolving Colab in-memory Pillow cache collision...')
    for mod in list(sys.modules.keys()):
        if mod.startswith('PIL'):
            del sys.modules[mod]
    try:
        from PIL import Image, ImageDraw
    except (ImportError, AttributeError):
        subprocess.run([sys.executable, '-m', 'pip', 'install', '--force-reinstall', '-q', 'pillow'])
        for mod in list(sys.modules.keys()):
            if mod.startswith('PIL'):
                del sys.modules[mod]
        from PIL import Image, ImageDraw

DATASET_TAR = 'dataset_distilled.tar.gz'
DISTILLED_DIR = Path('dataset_distilled')
JSONL_PATH = DISTILLED_DIR / 'curated_distilled_5k.jsonl'

# ─────────────────────────────────────────────────────────────────────────────
# Step 3.1: Download or Unpack Baseline Dataset (5,000 Everyday Garments)
# ─────────────────────────────────────────────────────────────────────────────
baseline_samples = []
if not JSONL_PATH.exists():
    if not os.path.exists(DATASET_TAR) or os.path.getsize(DATASET_TAR) < 5000000:
        print('📥 Downloading baseline distilled dataset (9.2 MB)...')
        import urllib.request
        MIRRORS = [
            'https://dressapp.co/api/v1/dataset/distilled',
            'https://github.com/Yoram-Jacobs/DressAppV1/raw/main/dataset_distilled.tar.gz',
            'https://raw.githubusercontent.com/Yoram-Jacobs/DressAppV1/main/dataset_distilled.tar.gz',
        ]
        downloaded = False
        for url in MIRRORS:
            try:
                print(f'Attempting download from {url}...')
                req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
                with urllib.request.urlopen(req, timeout=30) as resp, open(DATASET_TAR, 'wb') as out_f:
                    out_f.write(resp.read())
                if os.path.exists(DATASET_TAR) and os.path.getsize(DATASET_TAR) > 5000000:
                    print(f'✅ Downloaded {os.path.getsize(DATASET_TAR)} bytes successfully!')
                    downloaded = True
                    break
            except Exception as e:
                print(f'Mirror {url} failed: {e}')
        if not downloaded:
            print('Falling back to SCP from VPS...')
            os.system('scp -o StrictHostKeyChecking=no root@178.105.144.142:/srv/AI-Stylist/dataset_distilled.tar.gz .')

    if os.path.exists(DATASET_TAR) and os.path.getsize(DATASET_TAR) > 5000000:
        print('📦 Unpacking dataset_distilled.tar.gz...')
        with tarfile.open(DATASET_TAR, 'r:gz') as tar:
            tar.extractall('.')

if JSONL_PATH.exists():
    with open(JSONL_PATH, 'r', encoding='utf-8') as f:
        for line in f:
            if not line.strip(): continue
            rec = json.loads(line)
            img_rel_path = DISTILLED_DIR / rec['image_path']
            if img_rel_path.exists():
                baseline_samples.append({
                    'image_path': str(img_rel_path),
                    'system': rec['system_prompt'],
                    'user': rec['user_prompt'],
                    'assistant': json.dumps(rec['ground_truth'], ensure_ascii=False),
                    'lang': rec.get('target_language', 'en'),
                    'category': rec.get('category_target', 'Unknown'),
                    'sub_category': rec['ground_truth'].get('sub_category', 'General'),
                })
    print(f'✅ Loaded {len(baseline_samples)} baseline everyday garments!')
else:
    print('⚠️ Baseline dataset archive unavailable; proceeding with cultural attire dataset.')

# Directory for all cultural assets
CULTURAL_DIR = Path('cultural_dataset')
CULTURAL_IMG_DIR = CULTURAL_DIR / 'images'
CULTURAL_IMG_DIR.mkdir(parents=True, exist_ok=True)

ALL_LANGS = ["en", "he", "ar", "es", "fr", "de", "it", "pt", "nl", "ru", "zh", "ja", "hi"]

SYSTEM_PROMPT = (
    "Output raw JSON only ({...}). No markdown/intro.\n"
    "• Language: English keys/values/fabrics/colors.\n"
    "• name & title: 2-4 English words [Color] [Material/Cut] [Type]. Never generic ('Garment','Clothing').\n"
    "• category: 'Top'|'Bottom'|'Outerwear'|'Full Body'|'Footwear'|'Accessories'. Sweaters/tees='Top'. Outerwear=coats/jackets. Dresses/sets/robes/gowns='Full Body'.\n"
    "• sub_category: Specific cut ('T-Shirt','Sweater','Jeans','Pants','Skirt','Dresses','Suits','Galabiya','Kaftan','Thobe','Abaya','Kurta','Sherwani','Sari','Hanbok','Kimono','Dirndl','Guayabera','Oxfords','Loafers','Boots','Sandals','Sneakers','Heels','Pumps','Flats','Handbag','Sunglasses','Belts','Headwear','Scarves & Wraps').\n"
    "• Cultural & Traditional: Galabiya/Kaftan/Thobe/Abaya/Sari/Hanbok/Kimono/Dirndl -> category:'Full Body' (open Abaya/Kaftan -> 'Outerwear'). Kurta/Guayabera -> 'Top'. Sherwani -> 'Outerwear'|'Top'.\n"
    "• Traditional Headwear & Accessories: Kippah/Yarmulke, Keffiyeh/Shemagh, Turban/Dastar/Pagri, Hijab, Mitpachat/Tichel, Shtreimel, Fez -> sub_category:'Headwear' (or 'Scarves & Wraps' when draped as scarf), category:'Accessories'.\n"
    "• pattern: 'solid'|'printed'|'geometric'|'striped'|'plaid'|'floral'|'camouflage'|'embroidered'."
)

# ─────────────────────────────────────────────────────────────────────────────
# Step 3.2: Stream Real Photographic Cultural Garments from Hugging Face
# ─────────────────────────────────────────────────────────────────────────────
hf_cultural_samples = []

try:
    print('📥 Streaming real e-commerce cultural garments from ashraq/fashion-product-images-small (Myntra)...')
    from datasets import load_dataset
    hf_myntra = load_dataset('ashraq/fashion-product-images-small', split='train')
    
    CULTURAL_MYNTRA_MAP = {
        'Kurtas': ('Top', 'Kurta', 'smart-casual', 'Cotton'),
        'Kurtis': ('Top', 'Kurta', 'smart-casual', 'Cotton'),
        'Sarees': ('Full Body', 'Sari', 'formal', 'Silk'),
        'Kurta Sets': ('Full Body', 'Kurta', 'smart-casual', 'Silk'),
        'Dupatta': ('Accessories', 'Scarves & Wraps', 'smart-casual', 'Chiffon'),
        'Churidar': ('Bottom', 'Pants', 'casual', 'Cotton'),
        'Salwar': ('Bottom', 'Pants', 'casual', 'Cotton'),
        'Nehru Jackets': ('Outerwear', 'Sherwani', 'formal', 'Silk'),
        'Lehenga Choli': ('Full Body', 'Sari', 'formal', 'Brocade'),
    }
    
    myntra_count = 0
    for row in hf_myntra:
        art_type = row.get('articleType')
        if art_type in CULTURAL_MYNTRA_MAP:
            cat, sub_cat, dress_code, default_fabric = CULTURAL_MYNTRA_MAP[art_type]
            pil_img = row.get('image')
            if not pil_img:
                continue
            
            img_id = row.get('id', random.randint(10000, 99999))
            img_path = CULTURAL_IMG_DIR / f'hf_myntra_{img_id}.jpg'
            if not img_path.exists():
                pil_img.convert('RGB').save(img_path, 'JPEG', quality=88)
            
            color = row.get('baseColour') or 'Multi'
            raw_gender = str(row.get('gender') or '').lower()
            gender = 'men' if 'men' in raw_gender else ('women' if 'women' in raw_gender else 'unisex')
            raw_title = row.get('productDisplayName') or f'{color} {sub_cat}'
            pat = 'printed' if 'print' in raw_title.lower() else ('embroidered' if 'embroid' in raw_title.lower() else 'solid')
            
            target_lang = ALL_LANGS[myntra_count % len(ALL_LANGS)]
            
            gt = {
                'category': cat,
                'sub_category': sub_cat,
                'item_type': art_type,
                'name': f'{color} {sub_cat}',
                'title': raw_title,
                'caption': f'A {color.lower()} {sub_cat.lower()} crafted for traditional and {dress_code} wear.',
                'dress_code': dress_code,
                'gender': gender,
                'pattern': pat,
                'colors': [{'name': color, 'pct': 100}],
                'fabric_materials': [{'name': default_fabric, 'pct': 100}],
                'season': ['spring', 'summer', 'fall'],
                'tags': [sub_cat.lower(), 'traditional', 'ethnic', dress_code],
                'condition': 'excellent',
                'quality': 'mid',
                'state': 'new',
                'image_quality_status': 'complete',
                'image_quality_reason': None,
                'reconstruction_prompt': None
            }
            
            hf_cultural_samples.append({
                'image_path': str(img_path),
                'system': SYSTEM_PROMPT,
                'user': f'**OUTPUT LANGUAGE = {target_lang}**. Extract structured garment attributes. Return raw JSON.',
                'assistant': json.dumps(gt, ensure_ascii=False),
                'lang': target_lang,
                'category': cat,
                'sub_category': sub_cat,
            })
            myntra_count += 1
            if myntra_count >= 1200:
                break
                
    print(f'✅ Loaded {len(hf_cultural_samples)} real photographic South Asian items from Hugging Face!')
except Exception as e:
    print(f'ℹ️ Notice: Hugging Face e-commerce stream skipped or cached ({e}); using local cultural assets.')

# Load Moroccan Kaftan dataset if available
try:
    print('📥 Streaming real Moroccan Kaftan photographs from EDDOUM/moroccan_caftan...')
    from datasets import load_dataset
    hf_caftan = load_dataset('EDDOUM/moroccan_caftan', split='train')
    caftan_count = 0
    for row in hf_caftan:
        pil_img = row.get('image')
        if not pil_img: continue
        img_path = CULTURAL_IMG_DIR / f'hf_caftan_{caftan_count}.jpg'
        if not img_path.exists():
            pil_img.convert('RGB').save(img_path, 'JPEG', quality=88)
        
        target_lang = ALL_LANGS[caftan_count % len(ALL_LANGS)]
        gt = {
            'category': 'Full Body',
            'sub_category': 'Kaftan',
            'item_type': 'Moroccan Hand-Embroidered Kaftan',
            'name': 'Embroidered Silk Kaftan',
            'title': 'Moroccan Kaftan with Sfifa & Embroidery',
            'caption': 'An authentic Moroccan silk kaftan adorned with intricate embroidery and sfifa.',
            'dress_code': 'formal',
            'gender': 'women',
            'pattern': 'embroidered',
            'colors': [{'name': 'Gold', 'pct': 40}, {'name': 'Emerald Green', 'pct': 60}],
            'fabric_materials': [{'name': 'Silk', 'pct': 70}, {'name': 'Brocade', 'pct': 30}],
            'season': ['fall', 'winter', 'spring'],
            'tags': ['kaftan', 'traditional', 'embroidered', 'formal'],
            'condition': 'excellent',
            'quality': 'luxury',
            'state': 'new',
            'image_quality_status': 'complete',
            'image_quality_reason': None,
            'reconstruction_prompt': None
        }
        hf_cultural_samples.append({
            'image_path': str(img_path),
            'system': SYSTEM_PROMPT,
            'user': f'**OUTPUT LANGUAGE = {target_lang}**. Extract structured garment attributes. Return raw JSON.',
            'assistant': json.dumps(gt, ensure_ascii=False),
            'lang': target_lang,
            'category': 'Full Body',
            'sub_category': 'Kaftan',
        })
        caftan_count += 1
    print(f'✅ Loaded {caftan_count} real Moroccan Kaftan photos from Hugging Face!')
except Exception as e:
    print(f'ℹ️ Moroccan Kaftan stream notice: {e}')

# ─────────────────────────────────────────────────────────────────────────────
# Step 3.3: Targeted Distillation for Traditional Attire & Headwear
# ─────────────────────────────────────────────────────────────────────────────
CULTURAL_DISTILLED_ARCHETYPES = [
    # ── Traditional Robes & Apparel ──
    {
        "sub_category": "Galabiya",
        "category": "Full Body",
        "item_type": "Traditional Galabiya Robe",
        "gender": "unisex",
        "dress_code": "casual",
        "primary_fabrics": [("Cotton", 85), ("Linen", 15)],
        "patterns": ["solid", "embroidered", "striped"],
        "base_colors": [("White", (245, 245, 240)), ("Beige", (225, 215, 195)), ("Navy Blue", (25, 35, 75))],
        "names": {
            "en": "Traditional Embroidered Galabiya",
            "he": "ג'לביה מסורתית רקומה",
            "ar": "جلابية تقليدية مطرزة",
            "es": "Chilaba tradicional bordada",
            "fr": "Djellaba traditionnelle brodée",
            "de": "Traditionelle bestickte Dschellaba",
            "it": "Galabiya tradizionale ricamata",
            "pt": "Galabiya tradicional bordada",
            "nl": "Traditionele geborduurde galabiya",
            "ru": "Традиционная вышитая галабея",
            "zh": "传统刺绣加拉比亚长袍",
            "ja": "伝統的な刺繍ガラビーヤ",
            "hi": "पारंपरिक कढ़ाईदार जलाबिया",
        },
        "shape": "long_tunic",
    },
    {
        "sub_category": "Thobe",
        "category": "Full Body",
        "item_type": "Tailored Gulf Thobe",
        "gender": "men",
        "dress_code": "smart-casual",
        "primary_fabrics": [("Cotton", 90), ("Polyester", 10)],
        "patterns": ["solid"],
        "base_colors": [("Crisp White", (250, 250, 252)), ("Cream", (245, 240, 225))],
        "names": {
            "en": "Classic Tailored White Thobe",
            "he": "ת'וב מחויט מסורתי",
            "ar": "ثوب أبيض خليجي مفصل",
            "es": "Thobe clásico entallado",
            "fr": "Thobe classique sur mesure",
            "de": "Klassisches maßgeschneidertes Thobe",
            "it": "Thobe classico su misura",
            "pt": "Thobe clássico sob medida",
            "nl": "Klassieke getailleerde thobe",
            "ru": "Классический сшитый на заказ тоб",
            "zh": "经典定制阿拉伯白长袍",
            "ja": "クラシックなテーラードソーブ",
            "hi": "क्लासिक सिलवाया हुआ थोब",
        },
        "shape": "crisp_column",
    },
    {
        "sub_category": "Abaya",
        "category": "Full Body",
        "item_type": "Modest Open-Front Abaya",
        "gender": "women",
        "dress_code": "smart-casual",
        "primary_fabrics": [("Crepe", 80), ("Silk", 20)],
        "patterns": ["solid", "embroidered"],
        "base_colors": [("Classic Black", (20, 20, 25)), ("Dusty Rose", (185, 140, 145))],
        "names": {
            "en": "Modest Crepe Embroidered Abaya",
            "he": "עבאיה צנועה רקומה מקרפ",
            "ar": "عباية كريب أنيقة ومطرزة",
            "es": "Abaya elegante de crepé bordada",
            "fr": "Abaya élégante en crêpe brodé",
            "de": "Elegante bestickte Krepp-Abaya",
            "it": "Abaya elegante in crêpe ricamato",
            "pt": "Abaya elegante de crepe bordada",
            "nl": "Elegante geborduurde crêpe abaya",
            "ru": "Элегантная креповая вышитая абайя",
            "zh": "优雅刺绣双绉阿巴亚长袍",
            "ja": "エレガントなクレープ刺繍アバヤ",
            "hi": "सुरुचिपूर्ण क्रेप कढ़ाईदार अबाया",
        },
        "shape": "cloak_open",
    },
    {
        "sub_category": "Hanbok",
        "category": "Full Body",
        "item_type": "Traditional Korean Hanbok",
        "gender": "women",
        "dress_code": "formal",
        "primary_fabrics": [("Silk", 80), ("Ramie", 20)],
        "patterns": ["solid", "embroidered"],
        "base_colors": [("Pastel Pink & Jade", (235, 175, 185)), ("Cerulean & Coral", (80, 130, 195))],
        "names": {
            "en": "Traditional Korean Silk Hanbok",
            "he": "הנבוק קוריאני מסורתי ממשי",
            "ar": "هانبوك كوري تقليدي من الحرير",
            "es": "Hanbok coreano tradicional de seda",
            "fr": "Hanbok coréen traditionnel en soie",
            "de": "Traditioneller koreanischer Seiden-Hanbok",
            "it": "Hanbok coreano tradizionale in seta",
            "pt": "Hanbok coreano tradicional de seda",
            "nl": "Traditionele Koreaanse zijden hanbok",
            "ru": "Традиционный корейский шелковый ханбок",
            "zh": "传统韩国丝绸韩服",
            "ja": "伝統的な韓国の絹のチマチョゴリ（韓服）",
            "hi": "पारंपरिक कोरियाई रेशमी हानबोक",
        },
        "shape": "hanbok_chima",
    },
    {
        "sub_category": "Kimono",
        "category": "Full Body",
        "item_type": "Formal Silk Kimono with Obi",
        "gender": "women",
        "dress_code": "formal",
        "primary_fabrics": [("Silk", 90), ("Satin", 10)],
        "patterns": ["floral", "embroidered"],
        "base_colors": [("Cherry Blossom Indigo", (40, 50, 95)), ("Vermilion Orange", (215, 65, 40))],
        "names": {
            "en": "Formal Silk Kimono with Obi Sash",
            "he": "קימונו משי רשמי עם חגורת אובי",
            "ar": "كيمونو حريري رسمي مع حزام أوبي",
            "es": "Kimono formal de seda con faja obi",
            "fr": "Kimono formel en soie avec ceinture obi",
            "de": "Formeller Seidenkimono mit Obi-Gürtel",
            "it": "Kimono formale in seta con fascia obi",
            "pt": "Quimono formal de seda com faixa obi",
            "nl": "Formele zijden kimono met obi-band",
            "ru": "Формальное шелковое кимоно с поясом оби",
            "zh": "带腰带的和服正装",
            "ja": "帯付きの正装用絹着物",
            "hi": "ओबी बेल्ट के साथ रेशमी किमोनो",
        },
        "shape": "kimono_t",
    },
    {
        "sub_category": "Dirndl",
        "category": "Full Body",
        "item_type": "Bavarian Trachten Dirndl",
        "gender": "women",
        "dress_code": "smart-casual",
        "primary_fabrics": [("Cotton", 70), ("Linen", 30)],
        "patterns": ["floral", "plaid", "embroidered"],
        "base_colors": [("Alpine Forest Green", (45, 80, 50)), ("Berry Red", (155, 30, 50))],
        "names": {
            "en": "Bavarian Folk Dirndl Dress with Apron",
            "he": "שמלת דירנדל בווארית מסורתית עם סינר",
            "ar": "فستان ديرندل بافاري تقليدي مع مريلة",
            "es": "Vestido bávaro Dirndl tradicional con delantal",
            "fr": "Robe traditionnelle bavaroise Dirndl avec tablier",
            "de": "Traditionelles bayerisches Trachten-Dirndl mit Schürze",
            "it": "Abito tradizionale bavarese Dirndl con grembiule",
            "pt": "Vestido bávaro tradicional Dirndl com avental",
            "nl": "Traditionele Beierse dirndl jurk met schort",
            "ru": "Традиционный баварский дирндль с передником",
            "zh": "带围裙的传统巴伐利亚迪恩德尔连衣裙",
            "ja": "エプロン付きの伝統的なバイエルン・ディアンドル",
            "hi": "पारंपरिक बवेरियन डिर्नड्ल ड्रेस एप्रन के साथ",
        },
        "shape": "dirndl_bodice",
    },
    {
        "sub_category": "Guayabera",
        "category": "Top",
        "item_type": "Four-Pocket Linen Guayabera",
        "gender": "men",
        "dress_code": "smart-casual",
        "primary_fabrics": [("Linen", 100)],
        "patterns": ["embroidered", "solid"],
        "base_colors": [("Natural Linen White", (245, 245, 242)), ("Pastel Sky", (140, 185, 220))],
        "names": {
            "en": "Linen Four-Pocket Guayabera Shirt",
            "he": "חולצת גואיאברה מפשתן עם ארבעה כיסים",
            "ar": "قميص غوايابيرا كتان بأربعة جيوب",
            "es": "Guayabera clásica de lino de cuatro bolsillos",
            "fr": "Chemise guayabera en lin à quatre poches",
            "de": "Leinen-Guayabera-Hemd mit vier Taschen",
            "it": "Camicia guayabera in lino a quattro tasche",
            "pt": "Camisa guayabera de linho com quatro bolsos",
            "nl": "Linnen guayabera-overhemd met vier zakken",
            "ru": "Льняная рубашка гуаябера с четырьмя карманами",
            "zh": "四口袋亚麻瓜亚贝拉衬衫",
            "ja": "4ポケット付きのリネン・グアジャベーラシャツ",
            "hi": "चार जेब वाली लिनन गुआयाबेरा शर्ट",
        },
        "shape": "guayabera_shirt",
    },
    # ── Traditional Headwear & Religious Accessories ──
    {
        "sub_category": "Headwear",
        "category": "Accessories",
        "item_type": "Embroidered Velvet Kippah",
        "gender": "men",
        "dress_code": "formal",
        "primary_fabrics": [("Velvet", 80), ("Satin", 20)],
        "patterns": ["embroidered", "geometric", "solid"],
        "base_colors": [("Midnight Black", (20, 20, 25)), ("Navy Blue", (25, 35, 75)), ("Royal White", (248, 248, 250))],
        "names": {
            "en": "Traditional Embroidered Velvet Kippah",
            "he": "כיפת קטיפה רקומה מסורתית",
            "ar": "كيباه تقليدية مخملية مطرزة",
            "es": "Kipá tradicional de terciopelo bordada",
            "fr": "Kippa traditionnelle en velours brodé",
            "de": "Traditionelle bestickte Samt-Kippa",
            "it": "Kippah tradizionale in velluto ricamato",
            "pt": "Quipá tradicional de veludo bordado",
            "nl": "Traditionele geborduurde fluwelen keppeltje",
            "ru": "Традиционная вышитая бархатная кипа",
            "zh": "传统刺绣丝绒犹太小圆帽",
            "ja": "伝統的な刺繍ベルベット・キッパ",
            "hi": "पारंपरिक कढ़ाईदार मखमली किप्पा",
        },
        "shape": "kippah_dome",
    },
    {
        "sub_category": "Headwear",
        "category": "Accessories",
        "item_type": "Checkered Traditional Keffiyeh Scarf",
        "gender": "unisex",
        "dress_code": "casual",
        "primary_fabrics": [("Cotton", 90), ("Wool", 10)],
        "patterns": ["geometric", "plaid"],
        "base_colors": [("Black & White", (30, 30, 30)), ("Red & White", (170, 30, 35))],
        "names": {
            "en": "Checkered Traditional Keffiyeh Scarf",
            "he": "כאפייה מסורתית משובצת עם גדילים",
            "ar": "كوفية شماغ تقليدية منقوشة بالهدب",
            "es": "Keffiyeh tradicional a cuadros con borlas",
            "fr": "Keffieh traditionnel à carreaux et glands",
            "de": "Traditionelles kariertes Palästinensertuch (Keffiyeh)",
            "it": "Keffiyeh tradizionale a scacchi con nappe",
            "pt": "Keffiyeh tradicional xadrez com borlas",
            "nl": "Traditionele geruite keffiyeh sjaal met kwastjes",
            "ru": "Традиционная клетчатая куфия с кисточками",
            "zh": "传统黑白方格流苏开菲亚头巾",
            "ja": "タッセル付き伝統的チェック柄クーフィーヤ",
            "hi": "पारंपरिक चेक्ड कफ़िया दुपट्टा",
        },
        "shape": "keffiyeh_fold",
    },
    {
        "sub_category": "Headwear",
        "category": "Accessories",
        "item_type": "Pleated Silk Traditional Turban",
        "gender": "men",
        "dress_code": "formal",
        "primary_fabrics": [("Silk", 60), ("Cotton", 40)],
        "patterns": ["solid", "embroidered"],
        "base_colors": [("Saffron Gold", (225, 160, 45)), ("Royal Maroon", (120, 20, 40)), ("Pristine White", (248, 248, 252))],
        "names": {
            "en": "Pleated Traditional Silk Turban",
            "he": "טורבן מסורתי ממשי וכותנה",
            "ar": "عمامة تقليدية مطوية من الحرير",
            "es": "Turbante tradicional de seda plisado",
            "fr": "Turban traditionnel en soie plissée",
            "de": "Traditioneller plissierter Seidenturban",
            "it": "Turbante tradizionale in seta a pieghe",
            "pt": "Turbante tradicional de seda plissado",
            "nl": "Traditionele geplooide zijden tulband",
            "ru": "Традиционный плиссированный шелковый тюрбан",
            "zh": "传统丝绸打褶包头巾",
            "ja": "伝統的なプリーツ加工のシルクターバン",
            "hi": "पारंपरिक प्लीटेड रेशमी पगड़ी",
        },
        "shape": "turban_wrap",
    },
    {
        "sub_category": "Headwear",
        "category": "Accessories",
        "item_type": "Modest Chiffon Hijab Wrap",
        "gender": "women",
        "dress_code": "casual",
        "primary_fabrics": [("Chiffon", 85), ("Silk", 15)],
        "patterns": ["solid", "floral"],
        "base_colors": [("Mauve Taupe", (155, 130, 135)), ("Oat Milk Beige", (225, 215, 200)), ("Slate Charcoal", (60, 65, 75))],
        "names": {
            "en": "Modest Chiffon Hijab Headscarf",
            "he": "חיג'אב צנוע משיפון",
            "ar": "حجاب شيفون أنيق ومحتشم",
            "es": "Hiyab modesto de gasa",
            "fr": "Hijab modeste en mousseline de soie",
            "de": "Eleganter Chiffon-Hijab",
            "it": "Hijab modesto in chiffon",
            "pt": "Hijab modesto de chiffon",
            "nl": "Elegante chiffon hijab",
            "ru": "Элегантный шифоновый хиджаб",
            "zh": "优雅雪纺穆斯林希贾布头巾",
            "ja": "エレガントなシフォン・ヒジャブ",
            "hi": "सुरुचिपूर्ण शिफॉन हिजाब",
        },
        "shape": "hijab_drape",
    },
    {
        "sub_category": "Headwear",
        "category": "Accessories",
        "item_type": "Velvet Fur Shtreimel",
        "gender": "men",
        "dress_code": "formal",
        "primary_fabrics": [("Fur", 70), ("Velvet", 30)],
        "patterns": ["solid"],
        "base_colors": [("Dark Sable Brown", (45, 30, 20))],
        "names": {
            "en": "Traditional Velvet and Fur Shtreimel",
            "he": "שטריימל פרווה וקטיפה מסורתי",
            "ar": "قبعة شتريمل تقليدية من الفرو والمخمل",
            "es": "Shtreimel tradicional de piel y terciopelo",
            "fr": "Shtreimel traditionnel en fourrure et velours",
            "de": "Traditioneller Pelz-Shtreimel mit Samt",
            "it": "Shtreimel tradizionale in pelliccia e velluto",
            "pt": "Shtreimel tradicional de pele e veludo",
            "nl": "Traditionele bont- en fluwelen sjtrejmel",
            "ru": "Традиционный меховой штреймл с бархатом",
            "zh": "传统皮草天鹅绒犹太皮帽",
            "ja": "伝統的な毛皮とベルベットのシュトライメル",
            "hi": "पारंपरिक फर और मखमली श्ट्रीमल टोपी",
        },
        "shape": "shtreimel_cylinder",
    },
]

def render_garment_silhouette(filepath: Path, shape: str, rgb: tuple, pattern: str):
    img = Image.new("RGB", (448, 448), color=(240, 242, 245))
    draw = ImageDraw.Draw(img)

    if shape in ("long_tunic", "flared_robe", "crisp_column", "cloak_open"):
        body = [(140, 60), (308, 60), (348, 410), (100, 410)]
        draw.polygon(body, fill=rgb, outline=(30, 30, 30), width=2)
        draw.polygon([(140, 60), (60, 190), (100, 210), (160, 120)], fill=rgb, outline=(30, 30, 30), width=2)
        draw.polygon([(308, 60), (388, 190), (348, 210), (288, 120)], fill=rgb, outline=(30, 30, 30), width=2)
        draw.polygon([(190, 60), (224, 110), (258, 60)], fill=(240, 242, 245), outline=(30, 30, 30), width=2)
    elif shape == "guayabera_shirt":
        body = [(130, 70), (318, 70), (330, 340), (118, 340)]
        draw.polygon(body, fill=rgb, outline=(30, 30, 30), width=2)
        draw.polygon([(130, 70), (50, 180), (90, 200), (150, 120)], fill=rgb, outline=(30, 30, 30), width=2)
        draw.polygon([(318, 70), (398, 180), (358, 200), (298, 120)], fill=rgb, outline=(30, 30, 30), width=2)
        draw.polygon([(194, 70), (224, 115), (254, 70)], fill=(240, 242, 245), outline=(30, 30, 30), width=2)
        draw.line([(180, 80), (180, 330)], fill=(60, 60, 60), width=2)
        draw.line([(268, 80), (268, 330)], fill=(60, 60, 60), width=2)
        draw.rectangle([(140, 130), (175, 165)], outline=(40, 40, 40), width=2)
        draw.rectangle([(273, 130), (308, 165)], outline=(40, 40, 40), width=2)
        draw.rectangle([(140, 250), (175, 290)], outline=(40, 40, 40), width=2)
        draw.rectangle([(273, 250), (308, 290)], outline=(40, 40, 40), width=2)
    elif shape == "kippah_dome":
        # Dome-shaped skullcap
        draw.chord([(114, 150), (334, 330)], start=180, end=360, fill=rgb, outline=(30, 30, 30), width=2)
        draw.line([(114, 240), (334, 240)], fill=(30, 30, 30), width=2)
        if pattern == "embroidered":
            gold_thread = (220, 180, 50)
            draw.arc([164, 190, 284, 240], start=180, end=360, fill=gold_thread, width=2)
    elif shape == "keffiyeh_fold":
        # Folded square/triangle scarf with checkered pattern & fringe
        triangle = [(90, 130), (358, 130), (224, 350)]
        draw.polygon(triangle, fill=(245, 245, 245), outline=(30, 30, 30), width=2)
        # Checkered geometric markings
        pattern_color = rgb
        for x in range(120, 330, 22):
            draw.line([(x, 130), (224, 350)], fill=pattern_color, width=1)
        for y in range(150, 330, 24):
            x_left = 90 + int((y - 130) * (224 - 90) / (350 - 130))
            x_right = 358 - int((y - 130) * (358 - 224) / (350 - 130))
            draw.line([(x_left, y), (x_right, y)], fill=pattern_color, width=1)
    elif shape == "turban_wrap":
        # Pleated wrapped oval
        draw.ellipse([(114, 110), (334, 320)], fill=rgb, outline=(30, 30, 30), width=2)
        for y in range(150, 300, 30):
            draw.arc([(124, y - 20), (324, y + 40)], start=10, end=170, fill=(30, 30, 30), width=2)
    elif shape == "hijab_drape":
        # Oval face opening with draped shoulder mantle
        mantle = [(110, 140), (338, 140), (370, 380), (78, 380)]
        draw.polygon(mantle, fill=rgb, outline=(30, 30, 30), width=2)
        draw.ellipse([(174, 120), (274, 250)], fill=(240, 242, 245), outline=(30, 30, 30), width=2)
    elif shape == "shtreimel_cylinder":
        # Wide circular fur crown with dark velvet center
        draw.ellipse([(90, 160), (358, 290)], fill=(45, 30, 20), outline=(20, 15, 10), width=3)
        draw.ellipse([(154, 180), (294, 260)], fill=(15, 15, 20), outline=(20, 15, 10), width=2)
    else:
        body = [(150, 60), (298, 60), (360, 420), (88, 420)]
        draw.polygon(body, fill=rgb, outline=(30, 30, 30), width=2)
        draw.polygon([(150, 60), (70, 170), (105, 190), (170, 110)], fill=rgb, outline=(30, 30, 30), width=2)
        draw.polygon([(298, 60), (378, 170), (343, 190), (278, 110)], fill=rgb, outline=(30, 30, 30), width=2)
        draw.rectangle([(130, 180), (318, 220)], fill=(210, 170, 50), outline=(20, 20, 20), width=2)

    if pattern == "embroidered" and shape not in ("kippah_dome", "keffiyeh_fold"):
        gold_thread = (220, 180, 50)
        for y in range(90, 380, 24):
            draw.arc([204, y, 244, y + 16], start=0, end=180, fill=gold_thread, width=2)
            draw.line([(180, y), (200, y)], fill=gold_thread, width=2)
            draw.line([(248, y), (268, y)], fill=gold_thread, width=2)

    img.save(filepath, "JPEG", quality=90)

distilled_cultural_samples = []
print('Synthesizing high-fidelity cultural attire & traditional headwear training pairs...')
for arch in CULTURAL_DISTILLED_ARCHETYPES:
    for color_name, rgb in arch["base_colors"]:
        for pat in arch["patterns"]:
            img_filename = f"distilled_{arch['item_type'].lower().replace(' ', '_')}_{color_name.lower().replace(' ', '_')}_{pat}.jpg"
            img_path = CULTURAL_IMG_DIR / img_filename
            if not img_path.exists():
                render_garment_silhouette(img_path, arch["shape"], rgb, pat)

            for lang in ALL_LANGS:
                name_localized = arch["names"].get(lang, arch["names"]["en"])
                clean_color_name = color_name.split()[-1]

                ground_truth = {
                    "category": arch["category"],
                    "sub_category": arch["sub_category"],
                    "item_type": arch["item_type"],
                    "name": f"{clean_color_name} {arch['item_type']}",
                    "title": name_localized,
                    "caption": f"A {clean_color_name.lower()} {arch['item_type'].lower()} crafted with {pat} {arch['primary_fabrics'][0][0].lower()}.",
                    "dress_code": arch["dress_code"],
                    "gender": arch["gender"],
                    "pattern": pat,
                    "colors": [{"name": clean_color_name, "pct": 100}],
                    "fabric_materials": [{"name": f[0], "pct": f[1]} for f in arch["primary_fabrics"]],
                    "season": ["spring", "summer", "fall", "winter"] if arch["category"] == "Accessories" else (["spring", "summer"] if arch["dress_code"] != "formal" else ["fall", "winter", "spring"]),
                    "tags": [arch["sub_category"].lower(), "traditional", pat, arch["dress_code"]],
                    "condition": "excellent",
                    "quality": "premium",
                    "state": "new",
                    "image_quality_status": "complete",
                    "image_quality_reason": None,
                    "reconstruction_prompt": None
                }

                distilled_cultural_samples.append({
                    "image_path": str(img_path),
                    "system": SYSTEM_PROMPT,
                    "user": f"**OUTPUT LANGUAGE = {lang}**. Extract structured garment attributes. Return raw JSON.",
                    "assistant": json.dumps(ground_truth, ensure_ascii=False),
                    "lang": lang,
                    "category": arch["category"],
                    "sub_category": arch["sub_category"]
                })

print(f'✅ Synthesized {len(distilled_cultural_samples)} multilingual samples for cultural attire & headwear!')

# ─────────────────────────────────────────────────────────────────────────────
# Step 3.4: Combine All Streams & Export Final Curated Training File
# ─────────────────────────────────────────────────────────────────────────────
combined_dataset = baseline_samples + hf_cultural_samples + distilled_cultural_samples
random.seed(42)
random.shuffle(combined_dataset)

out_file = 'curated_dataset_v5.jsonl'
with open(out_file, 'w', encoding='utf-8') as f:
    for s in combined_dataset:
        f.write(json.dumps(s, ensure_ascii=False) + '\n')

print(f'🎉 Exported {len(combined_dataset)} total samples to {out_file} ({Path(out_file).stat().st_size / 1024 / 1024:.2f} MB)!')
print(f'• Baseline Western Garments: {len(baseline_samples)}')
print(f'• Real Hugging Face Cultural Garments: {len(hf_cultural_samples)}')
print(f'• Targeted Distilled Cultural Attire & Headwear: {len(distilled_cultural_samples)}')


## 🚀 Step 4: Fine-Tuning Qwen2.5-VL-3B with QLoRA & Prompt Loss Masking
Fine-tune `Qwen/Qwen2.5-VL-3B-Instruct` using 4-bit BitsAndBytes quantization and PEFT LoRA.
- Native dynamic resolution processor (`min_pixels = 128*28*28`, `max_pixels = 448*448`).
- Prompt loss masking (`labels[:prompt_len] = -100`) ensures gradient updates strictly optimize the structured JSON output.
- Target attention projection matrices (`q_proj`, `k_proj`, `v_proj`, `o_proj`, dimension 2048) for optimal CUDA kernel speedup without BitsAndBytes unaligned MLP kernel stalls.
- Preserves the frozen vision encoder while fine-tuning multimodal alignment and multilingual output fluency.


In [ ]:
import json
from PIL import Image
import torch
from transformers import (
    Qwen2_5_VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from qwen_vl_utils import process_vision_info

MODEL_ID = 'Qwen/Qwen2.5-VL-3B-Instruct'

# Verify GPU
if not torch.cuda.is_available():
    raise RuntimeError('❌ No GPU detected! In Colab, go to menu: Runtime -> Change runtime type -> Hardware accelerator: T4 or A100 GPU')

print(f'Loading Qwen2.5-VL processor for {MODEL_ID}...')
min_pixels = 128 * 28 * 28
max_pixels = 448 * 448
processor = AutoProcessor.from_pretrained(MODEL_ID, min_pixels=min_pixels, max_pixels=max_pixels)

# 4-bit NF4 Quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    bnb_4bit_use_double_quant=True,
)

print('Loading Qwen2.5-VL-3B-Instruct base model in 4-bit with SDPA...')
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    attn_implementation='sdpa',
    device_map='auto',
    torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
)

model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
)

# LoRA Configuration targeting attention projections
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)

model = get_peft_model(model, lora_config)
if hasattr(model, 'enable_input_require_grads'):
    model.enable_input_require_grads()
model.print_trainable_parameters()

# Dataset with ChatML formatting and exact Prompt Loss Masking
class QwenCulturalEyesDataset(torch.utils.data.Dataset):
    def __init__(self, jsonl_file, processor, max_samples=None):
        self.items = []
        with open(jsonl_file, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if line:
                    self.items.append(json.loads(line))
        if max_samples:
            self.items = self.items[:max_samples]
        self.processor = processor
        self.assistant_marker = processor.tokenizer.encode('<|im_start|>assistant\n', add_special_tokens=False)

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        item = self.items[idx]
        img_path = item['image_path']

        messages = [
            {'role': 'system', 'content': item['system']},
            {
                'role': 'user',
                'content': [
                    {'type': 'image', 'image': img_path},
                    {'type': 'text', 'text': item['user']},
                ],
            },
            {'role': 'assistant', 'content': item['assistant']},
        ]

        text_full = self.processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        image_inputs, video_inputs = process_vision_info(messages)

        full_inputs = self.processor(
            text=[text_full],
            images=image_inputs,
            videos=video_inputs,
            padding=False,
            return_tensors='pt',
        )

        input_ids = full_inputs['input_ids'].squeeze(0)
        labels = input_ids.clone()

        # Find the <|im_start|>assistant marker\n        input_list = input_ids.tolist()
        m_len = len(self.assistant_marker)
        prompt_boundary = len(input_list)
        for i in range(len(input_list) - m_len, -1, -1):
            if input_list[i : i + m_len] == self.assistant_marker:
                prompt_boundary = i + m_len
                break

        # Mask prompt & vision tokens (-100)
        labels[:prompt_boundary] = -100

        result = {k: v.squeeze(0) for k, v in full_inputs.items()}
        result['labels'] = labels
        return result

train_dataset = QwenCulturalEyesDataset('curated_dataset_v5.jsonl', processor, max_samples=6000)

# Batch collator
def collate_fn(batch):
    elem = batch[0]
    res = {}
    for k in elem:
        if k in ('input_ids', 'labels', 'attention_mask'):
            tensors = [d[k] for d in batch]
            pad_val = -100 if k == 'labels' else processor.tokenizer.pad_token_id
            res[k] = torch.nn.utils.rnn.pad_sequence(tensors, batch_first=True, padding_value=pad_val)
        elif isinstance(elem[k], torch.Tensor):
            res[k] = torch.stack([d[k] for d in batch])
        else:
            res[k] = [d[k] for d in batch]
    return res

import warnings
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

training_args = TrainingArguments(
    output_dir='output_qwen_eyes_lora_v5',
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    logging_steps=20,
    max_steps=800,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    save_strategy='steps',
    save_steps=100,
    save_total_limit=2,
    warmup_steps=50,
    dataloader_num_workers=0,
    report_to='none',
)

trainer = Trainer(
    model=model,
    train_dataset=train_dataset,
    data_collator=collate_fn,
    args=training_args,
)

print('Starting QLoRA Fine-Tuning on Qwen2.5-VL-3B (Cultural Attire + Everyday Garments)...')
print('⏱️ Expected duration: ~25-35 minutes (800 steps)')
try:
    trainer.train()
    print('✅ Fine-tuning completed successfully!')
except KeyboardInterrupt:
    print('\n⚠️ Training interrupted early by user. Saving current checkpoint...')
finally:
    out_adapter = Path('output_qwen_eyes_lora_v5/adapter')
    out_adapter.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(out_adapter)
    processor.save_pretrained(out_adapter)
    print(f'✅ LoRA v5 Adapter successfully saved to {out_adapter}!')


## ⚙️ Step 5: Merge Adapter & Quantize to GGUF
Merge the fine-tuned LoRA v5 adapter into the base `Qwen2.5-VL-3B-Instruct` model and quantize with `llama.cpp` to `Q4_K_M` (production target).


In [ ]:
import os, sys
from pathlib import Path
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from peft import PeftModel

# Bypass incompatible torchao check in PEFT
import peft.import_utils
peft.import_utils.is_torchao_available = lambda: False
try:
    import peft.tuners.lora.torchao
    peft.tuners.lora.torchao.is_torchao_available = lambda: False
except Exception:
    pass

MODEL_ID = 'Qwen/Qwen2.5-VL-3B-Instruct'
adapter_dir = 'output_qwen_eyes_lora_v5/adapter'
merged_dir = 'qwen2_5_vl_3b_lora_v5_merged'

print(f"Loading base model in FP16 for merging from {MODEL_ID}...")
base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map='cpu',
)

print(f"Merging LoRA adapter from {adapter_dir}...")
merged_model = PeftModel.from_pretrained(base_model, adapter_dir)
merged_model = merged_model.merge_and_unload()

print(f"Saving merged FP16 model to {merged_dir}...")
merged_model.save_pretrained(merged_dir)
processor = AutoProcessor.from_pretrained(MODEL_ID)
processor.save_pretrained(merged_dir)
print("✅ Merged model successfully saved!")

# Convert to GGUF using llama.cpp
print("🔄 Converting merged model to GGUF (FP16)...")
os.system(f"python llama.cpp/convert_hf_to_gguf.py {merged_dir} --outfile Qwen2.5-VL-3B-Instruct-v5-F16.gguf --outtype f16")

# Quantize to Q4_K_M
print("⚙️ Quantizing to Q4_K_M (production target)...")
os.system("./llama.cpp/build/bin/llama-quantize Qwen2.5-VL-3B-Instruct-v5-F16.gguf Qwen2.5-VL-3B-Instruct-v5-Q4_K_M.gguf Q4_K_M")

# Prepare production name
os.system("cp Qwen2.5-VL-3B-Instruct-v5-Q4_K_M.gguf Qwen2.5-VL-3B-Instruct-Q4_K_M.gguf")

gguf_path = Path("Qwen2.5-VL-3B-Instruct-Q4_K_M.gguf")
if gguf_path.exists():
    print(f"🎉 Production GGUF successfully built: {gguf_path.name} ({gguf_path.stat().st_size / 1024 / 1024:.1f} MB)")
else:
    raise RuntimeError("❌ GGUF quantization failed!")


## 🧪 Step 6: Cultural Attire Evaluation & Smoke Tests
Verify that the fine-tuned DressApp Eyes model accurately extracts:
1. `category: "Full Body"` (or `Top`/`Outerwear`) and exact cultural `sub_category` (`Galabiya`, `Kaftan`, `Thobe`, `Abaya`, `Kimono`, `Sari`, etc.).
2. Accurate recognition of `pattern: "embroidered"` and authentic fabric materials.
3. Idiomatic localized captions in Hebrew, Arabic, Hindi, Japanese, and Spanish.


In [ ]:
import json, glob, os
from PIL import Image
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

print('🧪 Loading merged model for evaluation...')
eval_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    'qwen2_5_vl_3b_lora_v5_merged',
    torch_dtype=torch.float16,
    device_map='auto',
)
processor = AutoProcessor.from_pretrained('qwen2_5_vl_3b_lora_v5_merged')

# Select test images across cultural archetypes & headwear
test_cases = [
    ('cultural_dataset/images/distilled_traditional_galabiya_robe_white_embroidered.jpg', 'Hebrew (עברית)', 'Galabiya in Hebrew'),
    ('cultural_dataset/images/distilled_embroidered_velvet_kippah_black_embroidered.jpg', 'Hebrew (עברית)', 'Velvet Kippah in Hebrew'),
    ('cultural_dataset/images/distilled_checkered_traditional_keffiyeh_scarf_white_geometric.jpg', 'Arabic (العربية)', 'Checkered Keffiyeh in Arabic'),
    ('cultural_dataset/images/distilled_pleated_traditional_silk_turban_gold_solid.jpg', 'Hindi (हिन्दी)', 'Silk Turban in Hindi'),
    ('cultural_dataset/images/distilled_modest_chiffon_hijab_wrap_taupe_solid.jpg', 'English', 'Chiffon Hijab in English'),
    ('cultural_dataset/images/distilled_linen_four-pocket_guayabera_shirt_white_embroidered.jpg', 'Spanish (Español)', 'Guayabera in Spanish'),
]

SYSTEM_PROMPT = (
    "Output raw JSON only ({...}). No markdown/intro.\n"
    "• Language: English keys/values/fabrics/colors.\n"
    "• name & title: 2-4 English words [Color] [Material/Cut] [Type]. Never generic ('Garment','Clothing').\n"
    "• category: 'Top'|'Bottom'|'Outerwear'|'Full Body'|'Footwear'|'Accessories'. Sweaters/tees='Top'. Outerwear=coats/jackets. Dresses/sets/robes/gowns='Full Body'.\n"
    "• sub_category: Specific cut ('T-Shirt','Sweater','Jeans','Pants','Skirt','Dresses','Suits','Galabiya','Kaftan','Thobe','Abaya','Kurta','Sherwani','Sari','Hanbok','Kimono','Dirndl','Guayabera','Oxfords','Loafers','Boots','Sandals','Sneakers','Heels','Pumps','Flats','Handbag','Sunglasses','Belts','Headwear','Scarves & Wraps').\n"
    "• pattern: 'solid'|'printed'|'geometric'|'striped'|'plaid'|'floral'|'camouflage'|'embroidered'."
)

for img_path, lang_name, label in test_cases:
    if not os.path.exists(img_path):
        continue
    print(f'\n============================================================')
    print(f'🔍 Test: {label} (Output Language: {lang_name})')
    print(f'============================================================')

    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {
            'role': 'user',
            'content': [
                {'type': 'image', 'image': img_path},
                {'type': 'text', 'text': f'**OUTPUT LANGUAGE: {lang_name}**. Return raw JSON.'},
            ],
        },
    ]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(text=[text], images=image_inputs, videos=video_inputs, return_tensors='pt').to(eval_model.device)

    with torch.no_grad():
        out = eval_model.generate(**inputs, max_new_tokens=400, temperature=0.1, do_sample=False)

    gen_text = processor.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    print(gen_text)
    try:
        parsed = json.loads(gen_text)
        print(f"✅ Extracted Category: {parsed.get('category')}")
        print(f"✅ Extracted Sub-Category: {parsed.get('sub_category')}")
        print(f"✅ Extracted Item Type: {parsed.get('item_type')}")
        print(f"✅ Extracted Title: {parsed.get('title')}")
    except Exception as e:
        print(f"JSON Parse info: {e}")


## 🚀 Step 7: 1-Click Secure Push to Hetzner VPS Server
Upload the fine-tuned `Qwen2.5-VL-3B-Instruct-Q4_K_M.gguf` to `/srv/AI-Stylist/eyes_gguf/` on the Hetzner VPS (`178.105.144.142`) and restart the `dressapp-eyes` container.


In [ ]:
import os
import subprocess
from pathlib import Path

gguf_file = 'Qwen2.5-VL-3B-Instruct-Q4_K_M.gguf'
assert Path(gguf_file).exists(), f'❌ {gguf_file} not found!'

# 1. Test SSH connection
print('Testing SSH connection to Hetzner VPS (178.105.144.142)...')
try:
    proc = subprocess.run(
        ['ssh', '-o', 'BatchMode=yes', '-o', 'ConnectTimeout=5', '-o', 'StrictHostKeyChecking=no', 'root@178.105.144.142', 'echo connected'],
        capture_output=True, text=True, timeout=8
    )
    is_connected = ('connected' in proc.stdout)
except Exception:
    is_connected = False

if is_connected:
    print('✅ SSH Key authorized! Uploading directly to VPS at datacenter speeds...')
    os.system(f'scp -o StrictHostKeyChecking=no {gguf_file} root@178.105.144.142:/srv/AI-Stylist/eyes_gguf/')
    print('🔄 Restarting dressapp-eyes container...')
    os.system("ssh -o StrictHostKeyChecking=no root@178.105.144.142 'cd /srv/AI-Stylist/deploy && docker compose restart eyes && sleep 8 && docker exec dressapp-eyes curl -s http://127.0.0.1:7860/healthz'")
    print('🎉 Model successfully deployed and healthy!')
else:
    print('ℹ️ Colab SSH Key is not authorized on the VPS.')
    print('You can download the GGUF file directly to your local computer:')
    try:
        from google.colab import files
        files.download(gguf_file)
        print('⬇️ Triggered browser download!')
    except Exception as e:
        print(f'Manual download helper: {e}')
